# Cleanup - Reset Demo Environment

This notebook removes all resources created by the ML Demo Pipeline:

| Resource Type | Examples |
|--------------|----------|
| **Model Monitors** | PATIENT_RISK_MODEL_MONITOR |
| **Alerts** | Drift detection alerts |
| **Tasks** | Retraining pipeline DAG tasks |
| **Inference Services** | SPCS model services |
| **Models** | PATIENT_RISK_MODEL (all versions) |
| **Feature Store** | Entity, Feature Views |
| **Views** | INFERENCE_LOGS_VIEW |
| **Dynamic Tables** | Feature Store DTs |
| **Tables** | RAW_PATIENT_DATA, TRAINING_FEATURES, etc. |
| **Stages** | MODEL_ARTIFACTS, JOB_PAYLOADS, etc. |
| **Compute Pool** | ML_DEMO_COMPUTE_POOL |
| **Warehouse** | ML_DEMO_WAREHOUSE |
| **Schema** | HEALTHCARE |
| **Database** | ML_DEMO_PIPELINE_DB |

**Warning**: This is destructive! Run cells selectively based on what you want to clean up.

## Setup

In [ ]:
%cd ..

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO)

from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
FULL_SCHEMA = f"{DB}.{SCHEMA}"
COMPUTE_WAREHOUSE = config.snowflake.warehouse
COMPUTE_POOL = config.compute.compute_pool
MODEL_NAME = config.model.model_name

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Database: {DB}")
print(f"Schema: {SCHEMA}")
print(f"Warehouse: {COMPUTE_WAREHOUSE}")
print(f"Compute Pool: {COMPUTE_POOL}")

## 1. Drop Alerts

Drop drift detection alerts that trigger retraining.

In [ ]:
alerts_result = session.sql(f"SHOW ALERTS IN SCHEMA {FULL_SCHEMA}").collect()
alerts = [row["name"] for row in alerts_result]

print(f"Found {len(alerts)} alerts")
for alert in alerts:
    try:
        session.sql(f"DROP ALERT IF EXISTS {FULL_SCHEMA}.{alert}").collect()
        print(f"  Dropped alert: {alert}")
    except Exception as e:
        print(f"  Could not drop alert {alert}: {e}")

## 2. Drop Tasks

Drop retraining pipeline tasks (DAG).

In [ ]:
tasks_result = session.sql(f"SHOW TASKS IN SCHEMA {FULL_SCHEMA}").collect()
tasks = [row["name"] for row in tasks_result]

print(f"Found {len(tasks)} tasks")
for task in tasks:
    try:
        session.sql(f"ALTER TASK IF EXISTS {FULL_SCHEMA}.{task} SUSPEND").collect()
        session.sql(f"DROP TASK IF EXISTS {FULL_SCHEMA}.{task}").collect()
        print(f"  Dropped task: {task}")
    except Exception as e:
        print(f"  Could not drop task {task}: {e}")

## 3. Drop Stored Procedures

Drop retraining and pipeline procedures.

In [ ]:
procs_result = session.sql(f"SHOW PROCEDURES IN SCHEMA {FULL_SCHEMA}").collect()
procs = [(row["name"], row["arguments"]) for row in procs_result]

print(f"Found {len(procs)} procedures")
for proc_name, args in procs:
    try:
        arg_types = args.split(" RETURN")[0].replace(proc_name, "").strip() if " RETURN" in args else "()"
        session.sql(f"DROP PROCEDURE IF EXISTS {FULL_SCHEMA}.{proc_name}{arg_types}").collect()
        print(f"  Dropped procedure: {proc_name}")
    except Exception as e:
        print(f"  Could not drop procedure {proc_name}: {e}")

## 4. Drop Model Monitors

Drop model monitoring configurations.

In [ ]:
MONITOR_NAME = f"{MODEL_NAME}_MONITOR"

try:
    session.sql(f"ALTER MODEL MONITOR IF EXISTS {FULL_SCHEMA}.{MONITOR_NAME} SUSPEND").collect()
    session.sql(f"DROP MODEL MONITOR IF EXISTS {FULL_SCHEMA}.{MONITOR_NAME}").collect()
    print(f"Dropped model monitor: {MONITOR_NAME}")
except Exception as e:
    print(f"Could not drop model monitor: {e}")

## 5. Delete Inference Services (SPCS)

Stop and delete model inference services.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session, database_name=DB, schema_name=SCHEMA)

try:
    model = registry.get_model(MODEL_NAME)
    versions = model.versions()
    
    for version in versions:
        version_name = version.version_name
        try:
            services = version.list_services().to_dict(orient="records")
            for svc in services:
                svc_name = svc.get("name") or svc.get("service_name")
                if svc_name:
                    print(f"Deleting service: {svc_name}")
                    version.delete_service(svc_name)
        except Exception as e:
            print(f"  Could not list/delete services for {version_name}: {e}")
    print("All inference services deleted")
except Exception as e:
    print(f"Could not access model registry: {e}")

## 6. Drop Registered Models

Delete all model versions and the model itself.

In [ ]:
try:
    model = registry.get_model(MODEL_NAME)
    registry.delete_model(MODEL_NAME)
    print(f"Deleted model: {MODEL_NAME}")
except Exception as e:
    print(f"Could not delete model: {e}")

## 7. Drop Feature Store Objects

Delete Feature Views and Entities.

In [ ]:
try:
    from snowflake.ml.feature_store import FeatureStore
    
    fs = FeatureStore(
        session=session, 
        database=DB, 
        name=SCHEMA, 
        default_warehouse=COMPUTE_WAREHOUSE
        )
    
    fvs = fs.list_feature_views().to_pandas()
    for _, row in fvs.iterrows():
        fv_name = row["NAME"]
        fv_version = row["VERSION"]
        try:
            fv = fs.get_feature_view(fv_name, fv_version)
            fs.delete_feature_view(fv)
            print(f"  Deleted feature view: {fv_name}/{fv_version}")
        except Exception as e:
            print(f"  Could not delete feature view {fv_name}: {e}")
    
    entities = fs.list_entities().to_pandas()
    for _, row in entities.iterrows():
        entity_name = row["NAME"]
        try:
            entity = fs.get_entity(entity_name)
            fs.delete_entity(entity_name)
            print(f"  Deleted entity: {entity_name}")
        except Exception as e:
            print(f"  Could not delete entity {entity_name}: {e}")
    
    print("Feature Store cleanup complete")
except Exception as e:
    print(f"Could not clean up Feature Store: {e}")

## 8. Drop Views

In [ ]:
VIEWS = [
    "INFERENCE_LOGS_VIEW",
]

views_result = session.sql(f"SHOW VIEWS IN SCHEMA {FULL_SCHEMA}").collect()
all_views = [row["name"] for row in views_result]

for view in VIEWS + all_views:
    try:
        session.sql(f"DROP VIEW IF EXISTS {FULL_SCHEMA}.{view}").collect()
        print(f"Dropped view: {view}")
    except Exception as e:
        print(f"Could not drop view {view}: {e}")

## 9. Drop Dynamic Tables

In [ ]:
dt_result = session.sql(f"SHOW DYNAMIC TABLES IN SCHEMA {FULL_SCHEMA}").collect()
dynamic_tables = [row["name"] for row in dt_result]

print(f"Found {len(dynamic_tables)} dynamic tables")
for dt in dynamic_tables:
    try:
        session.sql(f"DROP DYNAMIC TABLE IF EXISTS {FULL_SCHEMA}.{dt}").collect()
        print(f"  Dropped dynamic table: {dt}")
    except Exception as e:
        print(f"  Could not drop dynamic table {dt}: {e}")

## 10. Drop Tables

In [ ]:
TABLES = [
    "RAW_PATIENT_DATA",
    "STREAMING_PATIENT_DATA",
    "MODEL_METRICS",
    "BASELINE_PATIENT_DATA",
    "TEST_PATIENT_DATA",
    "ENGINEERED_FEATURES",
    "TRAINING_FEATURES",
    "TEST_FEATURES",
    "BASELINE_FOR_MONITORING",
    "MODEL_PREDICTIONS",
]

tables_result = session.sql(f"SHOW TABLES IN SCHEMA {FULL_SCHEMA}").collect()
all_tables = [row["name"] for row in tables_result]

for table in set(TABLES + all_tables):
    try:
        session.sql(f"DROP TABLE IF EXISTS {FULL_SCHEMA}.{table}").collect()
        print(f"Dropped table: {table}")
    except Exception as e:
        print(f"Could not drop table {table}: {e}")

## 11. Drop Stages

In [ ]:
STAGES = [
    "MODEL_ARTIFACTS",
    "JOB_PAYLOADS",
    "DATA_STAGE",
    "EVALUATION_RESULTS",
]

stages_result = session.sql(f"SHOW STAGES IN SCHEMA {FULL_SCHEMA}").collect()
all_stages = [row["name"] for row in stages_result]

for stage in set(STAGES + all_stages):
    try:
        session.sql(f"DROP STAGE IF EXISTS {FULL_SCHEMA}.{stage}").collect()
        print(f"Dropped stage: {stage}")
    except Exception as e:
        print(f"Could not drop stage {stage}: {e}")

## 12. (Optional) Drop Schema

**Warning**: This drops the entire schema and all remaining objects.

In [ ]:
# Uncomment to drop schema
# session.sql(f"DROP SCHEMA IF EXISTS {FULL_SCHEMA} CASCADE").collect()
# print(f"Dropped schema: {FULL_SCHEMA}")

## 13. (Optional) Drop Database

**Warning**: This drops the entire database.

In [ ]:
# Uncomment to drop database
# session.sql(f"DROP DATABASE IF EXISTS {DB} CASCADE").collect()
# print(f"Dropped database: {DB}")

## 14. (Optional) Stop and Drop Compute Pool

**Warning**: This stops all services using this compute pool and drops it.

In [ ]:
# Uncomment to drop compute pool
# try:
#     session.sql(f"ALTER COMPUTE POOL IF EXISTS {COMPUTE_POOL} STOP ALL").collect()
#     print(f"Stopped all services in compute pool: {COMPUTE_POOL}")
#     session.sql(f"DROP COMPUTE POOL IF EXISTS {COMPUTE_POOL}").collect()
#     print(f"Dropped compute pool: {COMPUTE_POOL}")
# except Exception as e:
#     print(f"Could not drop compute pool: {e}")

## 15. (Optional) Drop Warehouse

**Warning**: This drops the warehouse created for this demo.

In [ ]:
# Uncomment to drop warehouse
# session.sql(f"DROP WAREHOUSE IF EXISTS {WAREHOUSE}").collect()
# print(f"Dropped warehouse: {WAREHOUSE}")

## Verify Cleanup

Check what remains in the schema (if it still exists).

In [ ]:
try:
    print("Remaining objects in schema:")
    print("\nTables:")
    session.sql(f"SHOW TABLES IN SCHEMA {FULL_SCHEMA}").show()
    
    print("\nViews:")
    session.sql(f"SHOW VIEWS IN SCHEMA {FULL_SCHEMA}").show()
    
    print("\nDynamic Tables:")
    session.sql(f"SHOW DYNAMIC TABLES IN SCHEMA {FULL_SCHEMA}").show()
    
    print("\nStages:")
    session.sql(f"SHOW STAGES IN SCHEMA {FULL_SCHEMA}").show()
    
    print("\nTasks:")
    session.sql(f"SHOW TASKS IN SCHEMA {FULL_SCHEMA}").show()
    
    print("\nAlerts:")
    session.sql(f"SHOW ALERTS IN SCHEMA {FULL_SCHEMA}").show()
except Exception as e:
    print(f"Schema may have been dropped: {e}")

# Cleanup Complete!

All demo resources have been removed. To recreate the environment, start from **01_setup_infrastructure.ipynb**.